In [4]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv('clean_bank_reviews.csv')
print(df.columns)


Index(['review', 'rating', 'date', 'bank', 'source'], dtype='object')


In [12]:
from transformers import pipeline

# Initialize DistilBERT sentiment classifier
sentiment_analyzer = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

def get_sentiment(text):
    if not text:
        return {'label': 'NEUTRAL', 'score': 0.0}
    result = sentiment_analyzer(text, truncation=True, max_length=512)[0]
    return result

# Apply sentiment analysis
df['sentiment'] = df['processed_review'].apply(lambda x: get_sentiment(x)['label'])
df['sentiment_score'] = df['processed_review'].apply(lambda x: get_sentiment(x)['score'])
df['sentiment_label'] = df['sentiment'].map({'POSITIVE': 'positive', 'NEGATIVE': 'negative'}).fillna('neutral')

c:\10x AIMastery\10Academy-Week2-Fintech-Analytics\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


In [13]:
import spacy

nlp = spacy.load('en_core_web_sm')

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    return " ".join(tokens)

# Apply preprocessing if needed
if 'processed_review' not in df.columns:
    df['review'] = df['review'].fillna("").astype(str)
    df['processed_review'] = df['review'].apply(preprocess_text)

In [16]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
from transformers import pipeline

In [17]:
# Preprocessing (if needed)
if 'processed_review' not in df.columns:
    nlp = spacy.load('en_core_web_sm')
    def preprocess_text(text):
        if not isinstance(text, str):
            return ""
        doc = nlp(text.lower())
        tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
        return " ".join(tokens)
    df['review'] = df['review'].fillna("").astype(str)
    df['processed_review'] = df['review'].apply(preprocess_text)
    print("Created 'processed_review' column")

In [18]:
# Sentiment Analysis (if needed)
if 'sentiment_label' not in df.columns or 'sentiment_score' not in df.columns:
    sentiment_analyzer = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
    def get_sentiment(text):
        if not text:
            return {'label': 'NEUTRAL', 'score': 0.0}
        result = sentiment_analyzer(text, truncation=True, max_length=512)[0]
        return result
    df['sentiment'] = df['processed_review'].apply(lambda x: get_sentiment(x)['label'])
    df['sentiment_score'] = df['processed_review'].apply(lambda x: get_sentiment(x)['score'])
    df['sentiment_label'] = df['sentiment'].map({'POSITIVE': 'positive', 'NEGATIVE': 'negative'}).fillna('neutral')
    print("Completed sentiment analysis")

In [19]:
# Handle missing or non-string values in 'processed_review'
df['processed_review'] = df['processed_review'].fillna("").astype(str)

# TF-IDF Keyword Extraction
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=100)
tfidf_matrix = vectorizer.fit_transform(df['processed_review'])
feature_names = vectorizer.get_feature_names_out()

In [20]:
def get_top_keywords(text, vectorizer, n=5):
    tfidf_vector = vectorizer.transform([text])
    indices = tfidf_vector.toarray()[0].argsort()[-n:][::-1]
    return [feature_names[i] for i in indices]

df['keywords'] = df['processed_review'].apply(lambda x: get_top_keywords(x, vectorizer))
print(df[['processed_review', 'keywords']].head())

                                    processed_review  \
0                      app proactive good connection   
1                               send cbebirr app app   
2                                               good   
3                                         functional   
4  everytime uninstall app reach physically oldy ...   

                         keywords  
0      [good, app, ነው, በጣም, well]  
1      [send, app, wow, ነው, work]  
2      [good, ነው, wow, በጣም, well]  
3      [ነው, በጣም, wow, work, well]  
4  [app, security, wow, ነው, work]  


In [21]:
# Theme Clustering
themes = {
    'Account Access Issues': ['login error', 'password issue', 'account locked', 'login', 'access'],
    'Transaction Performance': ['slow transfer', 'transaction failed', 'loading time', 'transfer', 'slow'],
    'User Interface & Experience': ['good ui', 'user friendly', 'bad design', 'ui', 'interface'],
    'Customer Support': ['support unresponsive', 'helpdesk slow', 'support', 'helpdesk'],
    'Feature Requests': ['fingerprint login', 'bill payment', 'dark mode', 'fingerprint', 'payment']
}

def assign_themes(keywords):
    assigned_themes = []
    for theme, theme_keywords in themes.items():
        if any(keyword in theme_keywords for keyword in keywords):
            assigned_themes.append(theme)
    return assigned_themes if assigned_themes else ['Other']

df['themes'] = df['keywords'].apply(assign_themes)
print(df[['processed_review', 'keywords', 'themes']].head())

                                    processed_review  \
0                      app proactive good connection   
1                               send cbebirr app app   
2                                               good   
3                                         functional   
4  everytime uninstall app reach physically oldy ...   

                         keywords   themes  
0      [good, app, ነው, በጣም, well]  [Other]  
1      [send, app, wow, ነው, work]  [Other]  
2      [good, ነው, wow, በጣም, well]  [Other]  
3      [ነው, በጣም, wow, work, well]  [Other]  
4  [app, security, wow, ነው, work]  [Other]  


In [22]:
# Save Results
df['review_id'] = df.index
df[['review_id', 'review', 'sentiment_label', 'sentiment_score', 'themes']].to_csv('analysis_results.csv', index=False)
print("Results saved to analysis_results.csv")

# Summarize themes per bank
for bank in df['bank'].unique():
    bank_themes = df[df['bank'] == bank]['themes'].explode().value_counts()
    print(f"Themes for {bank}:\n{bank_themes}")

Results saved to analysis_results.csv
Themes for CBE:
themes
Other                      369
Transaction Performance      9
Account Access Issues        6
Feature Requests             3
Name: count, dtype: int64
Themes for BOA:
themes
Other                      384
Transaction Performance      7
Account Access Issues        7
Feature Requests             1
Name: count, dtype: int64
Themes for Dashen:
themes
Other                      369
Transaction Performance     15
Account Access Issues        8
Feature Requests             5
Name: count, dtype: int64
